# Use Case 3 - Glue ETL Job Notebook

## Converted from Glue Python job to Jupyter notebook

This notebook was generated from the original Glue ETL Python script to make the logic easier to teach and run step by step in a notebook.

### Use case
ETL for ML-ready data preparation and governed publishing.

### ETL purpose
Read churn data from S3, apply ETL-style cleanup and feature preparation, validate the dataset, and publish ML-ready outputs for SageMaker training and model versioning.

### How to teach this notebook
- Start with configuration and paths
- Run extraction first
- Inspect transformation logic
- Validate outputs before publish
- Explain how the same logic runs as a repeatable Glue job in production


## Notebook guidance

When running this in a notebook:
- replace AWS placeholders as needed
- inspect DataFrames after key transforms
- connect each step back to ETL principles: extract, transform, validate, load/publish


## Step 1 - Imports

Import the core Glue and PySpark libraries. `GlueContext` wraps `SparkContext` and provides Glue-specific readers and writers. `Job` tracks job bookmarks and commit state in the Glue service.


In [ ]:
import sys
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.utils import getResolvedOptions
from awsglue.job import Job
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType


## Step 2 - Initialise Glue context and resolve job arguments

`getResolvedOptions` reads named parameters passed to the Glue job at runtime (e.g. from a workflow trigger or the console). `job.init` registers the job run with the Glue service so bookmark state is tracked correctly.


In [ ]:
args = getResolvedOptions(sys.argv, ['JOB_NAME', 'SOURCE_PATH', 'TARGET_PATH'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)


## Step 3 - Extract: read raw CSV from S3

Read the raw churn CSV with headers enabled and schema inference on. Schema inference triggers an extra scan of the data but ensures numeric columns are typed correctly on read rather than requiring explicit casts for every field.


In [ ]:
df = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(args['SOURCE_PATH'])
)
# Quick shape check - Glue workers log this to CloudWatch
print(f"Rows: {df.count()}  Columns: {len(df.columns)}")


## Step 4 - Transform: clean types and engineer features

Apply all ETL transforms in a single chained expression to minimise intermediate Spark stages.

Key decisions:
- `TotalCharges` contains blank strings for new customers with no charges yet — replace with `null` before casting to `double` so downstream aggregations are null-safe.
- `label` encodes the target variable as `int` (1 = churn, 0 = retained) required by SageMaker built-in algorithms.
- `is_new_customer` flags tenure ≤ 6 months — a strong churn predictor.
- `monthly_charge_band` bins `MonthlyCharges` into Low / Medium / High — useful as a categorical feature and for segment reporting.
- `avg_monthly_spend_gap` captures billing irregularity: negative values indicate discounts or credits that may signal churn risk.


In [ ]:
from pyspark.ml.feature import Bucketizer

df = (
    df
    .withColumn(
        'TotalCharges',
        F.when(F.trim(F.col('TotalCharges')) == '', None)
         .otherwise(F.col('TotalCharges'))
         .cast(DoubleType())
    )
    .withColumn('MonthlyCharges', F.col('MonthlyCharges').cast(DoubleType()))
    .withColumn('tenure', F.col('tenure').cast(IntegerType()))
    .withColumn('label',
        F.when(F.col('Churn') == 'Yes', F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn('is_new_customer',
        F.when(F.col('tenure') <= 6, F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn('monthly_charge_band',
        F.when(F.col('MonthlyCharges') <= 35, F.lit('Low'))
         .when(F.col('MonthlyCharges') <= 70, F.lit('Medium'))
         .otherwise(F.lit('High'))
    )
    .withColumn(
        'avg_monthly_spend_gap',
        F.col('TotalCharges') - (F.col('tenure').cast(DoubleType()) * F.col('MonthlyCharges'))
    )
)


## Step 5 - Validate: check data quality before writing

Run lightweight checks before committing the write. If critical checks fail the job should raise an exception so the Glue workflow is marked as failed rather than silently writing bad data.


In [ ]:
null_labels = df.filter(F.col('label').isNull()).count()
null_total_charges = df.filter(F.col('TotalCharges').isNull()).count()
row_count = df.count()

print(f"Row count          : {row_count}")
print(f"Null labels        : {null_labels}")
print(f"Null TotalCharges  : {null_total_charges}")

if null_labels > 0:
    raise ValueError(f"Validation failed: {null_labels} rows have null label. Aborting job.")


## Step 6 - Load: write ML-ready output to S3 and commit the job

Write in CSV format with headers so SageMaker can read it directly. `mode('overwrite')` ensures idempotent reruns. `job.commit()` finalises the Glue job bookmark so re-runs only process new data.


In [ ]:
df.write.mode('overwrite').option('header', True).csv(args['TARGET_PATH'])
job.commit()
